In [ ]:
import sys
from pathlib import Path

print("Current dir:", Path.cwd())
sys.path.append(str(Path.cwd().parent))

import config as config
print("Loaded from:", config.__file__)

In [ ]:
from config import get_spark_session, s3_path, BUCKET_NAME
from pyspark.sql.functions import*
spark = get_spark_session("bronze-to-silver")

payments = spark.read.csv(
    s3_path("bronze", "order_payments", "olist_order_payments_dataset.csv"),
    header=True,
    inferSchema=True
)

payments.show(5)

Data profiling, understand the before any transformation

In [ ]:
payments.printSchema()

print(f"Number of records: {payments.count()}")

payments.show(10, truncate=False)

payments.describe().show()

from pyspark.sql.functions import col, count, when

payments.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in payments.columns
]).show()

In [ ]:
payments.count()

In [ ]:
total_rows = payments.count()

distinct_rows = payments.distinct().count()

print(f"Total Rows    : {total_rows}")
print(f"Distinct Rows : {distinct_rows}")
print(f"Duplicate Rows: {total_rows - distinct_rows}")

In [ ]:
payments.select("payment_type").distinct().show(truncate=False)

In [ ]:
payments.filter(col("payment_installments") == 0).show(truncate=False)

In [ ]:
payments.filter(
    col("order_id").isin(
        "744bade1fcf9ff3f31d860ace076d422",
        "1a57108394169c0b47d8f876acc9ba2d"
    )
).orderBy("order_id", "payment_sequential").show(truncate=False)

In [ ]:
from pyspark.sql.functions import col

payments.filter(col("payment_value") < 0).show()

In [ ]:
payments.filter(col("payment_value") == 0).show(truncate=False)